In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

# 1. Veri yükle
# df = pd.read_csv("/content/spellman_periodic_genes.txt") ## dosya yollarını kendi prjenize göre düzenleyiniz
df = pd.read_csv("/content/Spellman_with_labels.csv") # Updated file path
orf_col = df.columns[0]
time_cols = [col for col in df.columns if col not in [orf_col, 'label']]

X = df[time_cols].values
y = df['label'].values

# 2. Temel istatistikler + Fourier (FFT)
stats_features = pd.DataFrame({
    'mean': np.mean(X, axis=1),
    'std': np.std(X, axis=1),
    'max': np.max(X, axis=1),
    'min': np.min(X, axis=1),
    'range': np.max(X, axis=1) - np.min(X, axis=1),
    'skew': pd.Series(X.tolist()).apply(lambda x: pd.Series(x).skew()),
})

fft = np.abs(np.fft.fft(X, axis=1))
fft_power = np.sum(fft[:, 1:len(time_cols)//2], axis=1)
dominant_freq = np.argmax(fft[:, 1:len(time_cols)//2], axis=1)

features = pd.concat([stats_features, pd.DataFrame({'fft_power': fft_power, 'dominant_freq': dominant_freq})], axis=1)

# 3. Model Eğitimi & Kıyaslama
X_train, X_test, y_train, y_test = train_test_split(features, y, test_size=0.25, random_state=42, stratify=y)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    "XGBoost": xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss'),
    "SVM": SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', probability=True),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, target_names=['Negative (0)', 'Positive (1)']))



=== Random Forest ===
Accuracy: 0.9078
              precision    recall  f1-score   support

Negative (0)       0.92      0.97      0.95       938
Positive (1)       0.77      0.51      0.62       158

    accuracy                           0.91      1096
   macro avg       0.85      0.74      0.78      1096
weighted avg       0.90      0.91      0.90      1096


=== XGBoost ===
Accuracy: 0.9088
              precision    recall  f1-score   support

Negative (0)       0.93      0.97      0.95       938
Positive (1)       0.75      0.56      0.64       158

    accuracy                           0.91      1096
   macro avg       0.84      0.76      0.79      1096
weighted avg       0.90      0.91      0.90      1096


=== SVM ===
Accuracy: 0.7135
              precision    recall  f1-score   support

Negative (0)       0.95      0.70      0.81       938
Positive (1)       0.30      0.77      0.44       158

    accuracy                           0.71      1096
   macro avg       0.63 